In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

c:\Users\satya\Desktop\Agentic-Ai\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [8]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ---------------------------------------------------------
# 1. Create documents with metadata
# ---------------------------------------------------------

documents = [

    Document(
        page_content="""
        Machine learning allows computers to learn
        patterns from data.
        """,

        metadata={
            "source": "ml_notes.pdf",
            "page": 1,
            "topic": "Machine Learning"
        }
    ),

    Document(
        page_content="""
        Transformers use self-attention to capture
        relationships between words.
        """,

        metadata={
            "source": "nlp_notes.pdf",
            "page": 10,
            "topic": "Transformers"
        }
    )
]


# ---------------------------------------------------------
# 2. Split the documents into chunks
# ---------------------------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20
)

chunks = splitter.split_documents(documents)


# ---------------------------------------------------------
# 3. Add chunk-specific metadata
# ---------------------------------------------------------

for i, chunk in enumerate(chunks):

    # Add an ID to identify this particular chunk
    chunk.metadata["chunk_id"] = i


# ---------------------------------------------------------
# 4. Display the chunks
# ---------------------------------------------------------

for chunk in chunks:

    print("TEXT:")
    print(chunk.page_content)

    print("\nMETADATA:")
    print(chunk.metadata)

    print("-" * 50)

TEXT:
Machine learning allows computers to learn
        patterns from data.

METADATA:
{'source': 'ml_notes.pdf', 'page': 1, 'topic': 'Machine Learning', 'chunk_id': 0}
--------------------------------------------------
TEXT:
Transformers use self-attention to capture
        relationships between words.

METADATA:
{'source': 'nlp_notes.pdf', 'page': 10, 'topic': 'Transformers', 'chunk_id': 1}
--------------------------------------------------


Recursive chunking ⭐

This is the one you'll most commonly use in LangChain RAG.
1. \n\n       → paragraph
       ↓ too large?

2. \n         → line
       ↓ too large?

3. ". "       → sentence
       ↓ too large?

4. " "        → word
       ↓ too large?

5. ""         → character

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)
chunks = splitter.create_documents([text])
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ---")
    print(chunk.page_content)


--- Chunk 0 ---
Machine learning is a branch of artificial intelligence.
It allows computers to learn patterns from data.

Deep learning is a subset of machine learning.
It uses neural networks with multiple layers.

--- Chunk 1 ---
Transformers are neural network architectures
widely used in modern NLP systems.


In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
splitter = SemanticChunker(
    embeddings,

    # We use the distribution of semantic differences
    # between neighboring sentences.
    #
    # "percentile" means we identify unusually large
    # semantic differences using a percentile threshold.
    breakpoint_threshold_type="percentile",

    # 95 means we consider very large semantic differences
    # as possible points where a new chunk should begin.
    #
    # Higher value → fewer breakpoints → larger chunks
    # Lower value  → more breakpoints → smaller chunks
    breakpoint_threshold_amount=95
)

C:\Users\satya\AppData\Local\Temp\ipykernel_12572\3215615320.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings


# ---------------------------------------------------------
# 1. Create the embedding model
# ---------------------------------------------------------

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
query = "What are transformers?"
query_vec = embeddings.embed_query(query)

Store documents in Chroma — Vector Database
| Chunk                       | Embedding           | Metadata |
| --------------------------- | ------------------- | -------- |
| "RAG combines retrieval..." | `[0.12, 0.45, ...]` | page=1   |
| "Embeddings represent..."   | `[0.21, 0.31, ...]` | page=2   |
| "Vector databases..."       | `[0.91, 0.14, ...]` | page=3   |


In [1]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\satya\Desktop\Agentic-Ai\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\satya\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3827.09it/s]


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
loader = TextLoader("rag_document.txt")

documents = loader.load()

print("Number of documents:", len(documents))
print(documents[0].page_content)

Number of documents: 1
Retrieval-Augmented Generation (RAG)

Retrieval-Augmented Generation is a technique that combines information retrieval with large language models. Instead of relying only on the knowledge stored inside a language model, RAG retrieves relevant information from an external knowledge base and provides that information to the language model as context.

Document Chunking

Documents are usually too large to send directly to a language model. Therefore, documents are divided into smaller pieces called chunks. Chunking makes it easier to retrieve only the relevant parts of a document.

Recursive Character Text Splitting

Recursive character text splitting is a popular chunking technique. It tries to split the document using a hierarchy of separators such as paragraphs, new lines, spaces, and individual characters. This helps maintain meaningful pieces of text while keeping chunks within a specified size.

Embeddings

An embedding converts text into a numerical vector. 

In [8]:
chunks = splitter.split_documents(documents)

In [9]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

print("Chunks stored in Chroma!")

Chunks stored in Chroma!


In [ ]:
query = "How does RAG retrieve information?"

results = vectorstore.similarity_search(
    query,
    k=3
)
# results = vectorstore.similarity_search(
#     "How does retrieval work?",
#     k=3,
#     filter={"topic": "RAG"}
# )
print("\n\n========== RETRIEVED CHUNKS ==========")

for i, doc in enumerate(results):
    print("TEXT:", doc.page_content)
    print("METADATA:", doc.metadata)



========== RETRIEVED CHUNKS ==========
TEXT: A typical RAG pipeline contains several steps. First, a document is loaded. Next, the document is divided into chunks. Each chunk is converted into an embedding and stored in a vector database. When a user asks a question, the question is embedded and used to retrieve relevant chunks. Finally, the
METADATA: {'source': 'rag_document.txt'}
TEXT: Retrieval-Augmented Generation (RAG)
METADATA: {'source': 'rag_document.txt'}
TEXT: used to retrieve relevant chunks. Finally, the retrieved chunks are provided to a language model, which generates the final answer using the retrieved context.
METADATA: {'source': 'rag_document.txt'}


Document(
    page_content="Similarity search compares...",
    metadata={"source": "rag_document.txt"}
)

# Retriever

A **Retriever** takes a user query, searches the knowledge base, and returns the most relevant document chunks.

```text
User Query
    ↓
Retriever
    ↓
Chroma
    ↓
Relevant Chunks
```

### Without Retriever

```python
results = vectorstore.similarity_search(
    "What are embeddings?",
    k=3
)
```

```text
Your Code → Chroma → Top-3 Chunks
```

### With Retriever

```python
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

results = retriever.invoke(
    "What are embeddings?"
)
```

```text
Query
 ↓
Retriever
 ↓
Chroma
 ↓
Similarity Search
 ↓
Top-3 Chunks
```

### `as_retriever()`

```python
vectorstore.as_retriever()
```

means:

> Create a **retriever interface** around the vector store.

It **doesn't create another database or embedding model**.

### Key Difference

| Component         | Job                             |
| ----------------- | ------------------------------- |
| Embedding Model   | Text → Vector                   |
| Chroma            | Stores vectors + documents      |
| Similarity Search | Finds similar chunks            |
| Retriever         | Interface for retrieving chunks |
| LLM               | Generates final answer          |


In [16]:
retriver = vectorstore.as_retriever(search_kwargs={"k": 3})

In [23]:
results = retriver.invoke("What are embeddings?")
print(results[0].page_content)

Embeddings

An embedding converts text into a numerical vector. Texts with similar meanings tend to have vectors that are close to each other in the embedding space. Embeddings allow a system to perform semantic search rather than simply matching exact keywords.

Vector Databases


In [24]:
results = vectorstore.similarity_search_with_score(
    query,
    k=3
)

for doc, score in results:
    print("Score:", score)
    print("Text:", doc.page_content)

Score: 0.7163675427436829
Text: A typical RAG pipeline contains several steps. First, a document is loaded. Next, the document is divided into chunks. Each chunk is converted into an embedding and stored in a vector database. When a user asks a question, the question is embedded and used to retrieve relevant chunks. Finally, the
Score: 1.0029860734939575
Text: Retrieval-Augmented Generation (RAG)
Score: 1.0531949996948242
Text: used to retrieve relevant chunks. Finally, the retrieved chunks are provided to a language model, which generates the final answer using the retrieved context.


### What does MMR do?

**MMR = Maximal Marginal Relevance**

It balances:

* **Relevance** → How relevant is the chunk to the query?
* **Diversity** → How different is it from already selected chunks?

**Goal:** Avoid retrieving multiple chunks that contain the same information.

> **MMR = Relevant + Diverse results, less redundancy.**


In [25]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3,
        "fetch_k": 10
    }
)

In [26]:
results = retriver.invoke("What is rag?")
print(results[0].page_content)

A typical RAG pipeline contains several steps. First, a document is loaded. Next, the document is divided into chunks. Each chunk is converted into an embedding and stored in a vector database. When a user asks a question, the question is embedded and used to retrieve relevant chunks. Finally, the


### Hybrid Search

```text
                    Query
                      ↓
             ┌────────┴────────┐
             ↓                 ↓
        BM25 Search       Vector Search
             ↓                 ↓
     Keyword results     Semantic results
             ↓                 ↓
             └────────┬────────┘
                      ↓
              Combine Results
                      ↓
               Final Results
```

* **BM25** → finds results using **keyword matching**.
* **Vector Search** → finds results using **semantic meaning**.
* **Combine** → merges both results to improve retrieval.


```text
BM25 Results       Vector Results
     ↓                  ↓
     └───────┬──────────┘
             ↓
            RRF
             ↓
       Combined Ranking
```

### Simple Definition

> **RRF combines ranked results from multiple retrieval methods into one ranking, giving higher importance to documents that rank highly across multiple methods.**
